Section 1 - Load Data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")
RAW_DIR = BASE_DIR / "data" / "raw"

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")
stores = pd.read_csv(RAW_DIR / "stores.csv")
oil = pd.read_csv(RAW_DIR / "oil.csv")
holidays = pd.read_csv(RAW_DIR / "holidays_events.csv")
transactions = pd.read_csv(RAW_DIR / "transactions.csv")

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

Section 2 - Date Features

In [2]:
for df in [train, test]:

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["quarter"] = df["date"].dt.quarter
    df["week"] = df["date"].dt.isocalendar().week
    df["day_of_week"] = df["date"].dt.dayofweek

    df["is_weekend"] = (
        df["day_of_week"] >= 5
    ).astype(int)

Section 3 - Merge Store Information

In [3]:
train = train.merge(
    stores,
    on="store_nbr",
    how="left"
)

test = test.merge(
    stores,
    on="store_nbr",
    how="left"
)

Section 4 - Check Result

In [4]:
print(train.shape)
print(test.shape)

train.head()

(3000888, 17)
(28512, 16)


,id,date,store_nbr,family,sales,onpromotion,year,month,day,quarter,week,day_of_week,is_weekend,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,1,1,0,Quito,Pichincha,D,13
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,1,1,0,Quito,Pichincha,D,13
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,1,1,0,Quito,Pichincha,D,13
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,1,1,0,Quito,Pichincha,D,13
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,1,1,0,Quito,Pichincha,D,13


Section 5 — Merge Oil Prices

In [5]:
oil["date"] = pd.to_datetime(oil["date"])

train = train.merge(oil, on="date", how="left")
test = test.merge(oil, on="date", how="left")

print(train.shape)
print(test.shape)

train[["date", "dcoilwtico"]].head()

train["dcoilwtico"] = train["dcoilwtico"].ffill().bfill()
test["dcoilwtico"] = test["dcoilwtico"].ffill().bfill()

print(train["dcoilwtico"].isnull().sum())
print(test["dcoilwtico"].isnull().sum())

(3000888, 18)
(28512, 17)
0
0


Section 6 - Payday Features

In [7]:
for df in [train, test]:
    df["is_payday"] = (
        (df["day"] == 15) |
        (df["date"] == df["date"] + pd.offsets.MonthEnd(0))
    ).astype(int)

print(train.shape)
print(test.shape)

train[["date", "day", "dcoilwtico", "is_payday"]].head(20)

(3000888, 19)
(28512, 18)


,date,day,dcoilwtico,is_payday
0,2013-01-01,1,93.14,0
1,2013-01-01,1,93.14,0
2,2013-01-01,1,93.14,0
3,2013-01-01,1,93.14,0
4,2013-01-01,1,93.14,0
5,2013-01-01,1,93.14,0
6,2013-01-01,1,93.14,0
7,2013-01-01,1,93.14,0
8,2013-01-01,1,93.14,0
9,2013-01-01,1,93.14,0


Section 7 - Holiday Features

In [8]:
# Convert holiday dates
holidays["date"] = pd.to_datetime(holidays["date"])

# Keep only holidays that were not transferred
valid_holidays = holidays[holidays["transferred"] == False].copy()

# Create simple national holiday indicator
national_holidays = (
    valid_holidays[valid_holidays["locale"] == "National"]
    [["date"]]
    .drop_duplicates()
)

national_holidays["is_national_holiday"] = 1

# Merge with train and test
train = train.merge(national_holidays, on="date", how="left")
test = test.merge(national_holidays, on="date", how="left")

train["is_national_holiday"] = train["is_national_holiday"].fillna(0).astype(int)
test["is_national_holiday"] = test["is_national_holiday"].fillna(0).astype(int)

print(train.shape)
print(test.shape)

train[["date", "is_national_holiday"]].head(20)

(3000888, 20)
(28512, 19)


,date,is_national_holiday
0,2013-01-01,1
1,2013-01-01,1
2,2013-01-01,1
3,2013-01-01,1
4,2013-01-01,1
5,2013-01-01,1
6,2013-01-01,1
7,2013-01-01,1
8,2013-01-01,1
9,2013-01-01,1


Section 8 - Earthquake Feature

In [9]:
earthquake_date = pd.Timestamp("2016-04-16")

for df in [train, test]:
    df["earthquake_period"] = (
        (df["date"] >= earthquake_date) &
        (df["date"] <= earthquake_date + pd.Timedelta(days=30))
    ).astype(int)

print(train.shape)
print(test.shape)

train[["date", "earthquake_period"]].drop_duplicates().query(
    "date >= '2016-04-10' and date <= '2016-04-25'"
)

(3000888, 21)
(28512, 20)


,date,earthquake_period
2124144,2016-04-10,0
2125926,2016-04-11,0
2127708,2016-04-12,0
2129490,2016-04-13,0
2131272,2016-04-14,0
2133054,2016-04-15,0
2134836,2016-04-16,1
2136618,2016-04-17,1
2138400,2016-04-18,1
2140182,2016-04-19,1


Section 9 - Save Processed Data

In [10]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
test.to_csv(PROCESSED_DIR / "test_features.csv", index=False)

print("Saved processed train and test files.")
print(train.shape)
print(test.shape)

Saved processed train and test files.
(3000888, 21)
(28512, 20)
